In [ ]:
import os
import sys

import logging

%pwd
os.chdir("../")

# Add the absolute path to src/ so Python can find automatch
src_path = os.path.abspath("src")
if src_path not in sys.path:
    sys.path.append(src_path)
    
%pwd


from src.onBench.utils.EDA import *


logging.getLogger("snowflake").setLevel(logging.WARNING)
logging.getLogger("snowflake.connector").setLevel(logging.WARNING)
logging.getLogger("snowflake.snowpark").setLevel(logging.WARNING)


from src.onBench.entity.config_entity import DataIngestionConfig
from src.onBench.config.configuration import ConfigurationManager



c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\holoviews\__init__.py:81: FutureWarning: param.version.Version has been deprecated and will be removed in a future version.
  __version__ = str(param.version.Version(fpath=__file__, archive_commit="$Format:%h$",
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\param\version.py:200: FutureWarning: param.version.run_cmd has been deprecated and will be removed in a future version.
  remotes = run_cmd([cmd, 'remote', '-v'],
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\panel\config.py:35: FutureWarning: param.version.Version has been deprecated and will be removed in a future version.
  __version__ = str(param.version.Version(
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\site-packages\param\version.py:200: FutureWarning: param.version.run_cmd has been deprecated and will be removed in a future version.
  remotes = run_cmd([cmd, 'remote', '-v'],
c:\Users\fiscarelli\conda_venvs\onbench_venv\lib\

In [ ]:
from src.onBench.utils.snowflake_utils import get_snowpark_session
session = get_snowpark_session()

In [ ]:
df = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_CANDIDATE_CLEANED--_ALL_EDA
                     --where status = 'No mission'
                 """).to_pandas()



[2026-10-07 11:16:10,200: ERROR: ocsp_snowflake: WARNING!!! Using fail-open to connect. Driver is connecting to an HTTPS endpoint without OCSP based Certificate Revocation checking as it could not obtain a valid OCSP Response to use from the CA OCSP responder. Details: 
 {'driver': 'PythonConnector', 'version': '3.7.0', 'eventType': 'RevocationCheckFailure', 'eventSubType': 'OCSPResponseAttachedCertInvalid', 'sfcPeerHost': 'iojcjrsfcb1stg.blob.core.windows.net', 'certId': 'MEowBwYFKw4DAhoEFDRxGqvIMaRG9VdxNofHFsCKo4HoBBS4LzOmfFFPfx8t1cgVQjiksg6PLwITQQESwj7Yh2sdJjaiRAAAARLCPg==', 'ocspRequestBase64': <asn1crypto.ocsp.OCSPRequest 2754761964032 b'0R0P0N0L0J0\x07\x06\x05+\x0e\x03\x02\x1a\x04\x144q\x1a\xab\xc81\xa4F\xf5Wq6\x87\xc7\x16\xc0\x8a\xa3\x81\xe8\x04\x14\xb8/3\xa6|QO\x7f\x1f-\xd5\xc8\x15B8\xa4\xb2\x0e\x8f/\x02\x13A\x01\x12\xc2>\xd8\x87k\x1d&6\xa2D\x00\x00\x01\x12\xc2>'>, 'ocspResponderURL': 'http://oneocsp.microsoft.com/ocsp', 'errorMessage': '254009: Failed to verify the signature',

In [ ]:
df["CCNL_CODE"] = df["CCNL_CODE"].astype("Int64").astype("string")

df[df.select_dtypes(include=["object", "string"]).columns] = (
    df.select_dtypes(include=["object", "string"]).apply(lambda col: col.str.strip())
)


In [ ]:
keywords = [

    #non utilizzare
    #"curriculum", "cv", "vitae", 

    #generali
    "esperienza", "lavoro", "occupazione", "posizione", "ruolo", "azienda",
    "experience", "job", "position"

    # ruoli
    "addetto", "operaio", "impiegato", "responsabile", "cassiere", "magazziniere",
    "analista", "sviluppatore", "developer", "autista", "segretaria", "tecnico", "consulente",
    "worker", "manager", "coordinator", "assistant", "engineer", "specialist", "warehouse", "driver", "operator", "supervisor", "administrator", "consultant"

    # titoli di studio
    "diploma", "laurea", "master", "istituto", "universit",
    "degree", "master", "university", "school", "college",

    # competenze
    "excel", "office", "inglese", "informatica", "patente",
    "english", "computer", "driving license"

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "srl", "spa", "cooperativa", "comune", "regione"
    "regione", "municipality", "company", "organization", "corporation"
]

df["IS_VALID_CV"] = df["CURRICULUM"].str.lower().apply(
    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False
)

is_valid_cv = np.sum(df["IS_VALID_CV"])/df.shape[0]

print(f"""Percentage of invalid CVs (custom keywords matching): {1-is_valid_cv:.2%}""")




Percentage of invalid CVs (custom keywords matching): 46.78%


# MISSING CVs #

Other than using CV length to identify real CVs (50 characters), I also define a set of keywords that are normally found in a CV and I check whether a CV contains any. ù
So, if the CV contains at least 50 characters and at least one of the keywords mentioned, I classify it as real.
According to this method, 47% of candidate CS are not valid (are emtpy or contain meaningless text)

In [ ]:
general_info(df)


Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   CANDIDATE_ID_BH           451 non-null    int32         
 1   SALARY                    6 non-null      float64       
 2   DATE_AVAILABLE            393 non-null    datetime64[ns]
 3   WILL_RELOCATE             451 non-null    bool          
 4   START_DATE                451 non-null    datetime64[ns]
 5   END_DATE                  451 non-null    datetime64[ns]
 6   DAYS_ONBENCH_LAST         149 non-null    float64       
 7   DAYS_ONBENCH_TOT          451 non-null    int16         
 8   DAYS_ONBENCH_PERC         451 non-null    int8          
 9   PARTTIME_PREFERENZA_PERC  451 non-null    int8          
 10  TURNO_PREFERENZA          2 non-null      object        
 11  PARTTIME_PERC             451 non-null    float64       
 12  CANDIDATE_ID_PB

# COMMENTS on missing/unique values #

9085 candidates

Comments on candidate missing data (Bullhorn):
-   salary (99% missing)
-   turno_preferenza (99.5% missing)
-   parttime_preferenza_perc (99.7% missing)
-   skills (82% missing)
-   education (66% missing)
-   certifications (100% missing)
-   curriculum 48% missing
-   description (34% missing. this means that curriculum, skills, education and certifiations are all simultaneaously missing)

Comments on candidate missing data (PowerBase):
-   parttime_perc (first job) (88% missing)
-   CCNL level (first job) (0% missing)
-   CCNL code (first job) (1.4%)
-   CCNL description (first job) (1.4%)
-   legal_category (first job) (0% missing)


Some comments on unique values:
- 11 unique values for education
- 776 unique first jobs
- 172 unique CCNL codes
- 209 unique CCNL descriptions
- 6 unique legal categories

In [ ]:
columns_to_check = ["EDUCATION", "CERTIFICATIONS"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)


Unique values in 'EDUCATION':
['diploma' 'laurea' 'laurea breve' 'non specificato' 'scuola media'
 'scuola secondaria' 'universita' None]
----------------------------------------
Unique values in 'CERTIFICATIONS':
[None]
----------------------------------------


In [ ]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)



Unique values in 'CCNL_LEVEL':
['1' '2' '2tr' '3' '3 ap liv. 1' '4' '4 junior' '4 senior'
 '4 super (ex 5)' '4/1' '5' '5 (ex 6)' '5 fascia 1' '6' '6 junior'
 '6 senior' '6 super (ex 7)' '7' '8' 'a' 'ae1' 'ae2' 'ap prima'
 'area 3 liv. 1' 'area operatori' 'area3liv.1' 'b' 'b1' 'b2' 'bs super'
 'c' 'c/1' 'c/2' 'c1' 'c2' 'c3' 'comune' 'd' 'd/1' 'd/2' 'd1' 'd1 (ex d2)'
 'd2' 'd3' 'e' 'e1' 'e2' 'e3' 'e4' 'f' 'g' 'g1' 'h' 'i' 'op centrato'
 'prof.s. e funz.' 'v']
----------------------------------------
Unique values in 'CCNL_CODE':
<StringArray>
['1020',  '106', '1270',  '130', '1380', '1390', '1620',   '17', '1790',
 '1810', '1820', '1870', '1930', '2070',  '220', '2220',  '230', '2310',
 '2360',  '240', '2470', '2630',  '330',   '38', '3980',  '530', '5750',
  '700',  '710',  '720',   '88',   '89', '9013', '9019', '9050', '9275',
 '9332', '9335', '9354', '9357', '9408', '9504', '9633', '9646', '9662',
 '9669', '9674', '9683', '9719', '9726', '9746', '9776', '9791', '9796',
 '9816', '9817'

In [ ]:
columns_to_check = ["JOB_FIRST"]

# Print unique values for each column
for col in columns_to_check:
    print(f"Unique values in '{col}':")
    print(df[col].sort_values().unique())
    print("-" * 40)

Unique values in 'JOB_FIRST':
['add. rifinitura- qualità e confez. pelletteria' 'addetti global custody'
 'addetto agli allestimenti interni ferroviari' 'addetto agli scaffali'
 'addetto ai controlli contabili' 'addetto al magazzino'
 'addetto al magazzino settore abbigliamento' 'addetto al montaggio'
 'addetto alimentazione mulini' 'addetto alla composizione'
 'addetto alla fatturazione' 'addetto alla fresatrice'
 'addetto alla logistica' 'addetto alla pescheria'
 'addetto alla pianificazione acquisti'
 'addetto alla pressa per lo stampaggio'
 'addetto alla produzione di gomme da masticare'
 'addetto alla produzione di pasta alimentare'
 'addetto alla produzione di semiconduttori' 'addetto alla reception'
 'addetto alla vendita attrezzature audio video'
 'addetto alla verniciatura a spruzzo' 'addetto alle pulizie'
 'addetto alle pulizie negli stabili' 'addetto allo sportello bancario'
 'addetto all’assemblaggio di macchinari industriali'
 'addetto all’estrusione per materie plastiche'

In [ ]:
b1 = bar_plot(df, 'EDUCATION', rotation=25)
b2 = bar_plot(df, 'CERTIFICATIONS', rotation=25)
b3 = bar_plot(df, 'IS_VALID_CV', rotation=25)
b4 = bar_plot(df, 'CANDIDATE_STATUS', rotation=25)


(b1 + b2 + b3 + b4).cols(1)

:Layout
   .Bars.I   :Bars   [EDUCATION]   (COUNT)
   .Bars.II  :Bars   [CERTIFICATIONS]   (COUNT)
   .Bars.III :Bars   [IS_VALID_CV]   (COUNT)
   .Bars.IV  :Bars   [CANDIDATE_STATUS]   (COUNT)

In [ ]:
b1 = bar_plot(df, 'CAP', rotation=25)
b2 = bar_plot(df, 'CITY', rotation=25)
b3 = bar_plot(df, 'PROVINCE', rotation=25)

(b1 + b2 + b3).cols(1)

:Layout
   .Bars.I   :Bars   [CAP]   (COUNT)
   .Bars.II  :Bars   [CITY]   (COUNT)
   .Bars.III :Bars   [PROVINCE]   (COUNT)

In [ ]:
b1 = bar_plot(df, 'CCNL_LEVEL')#, label_range=30)
b2 = bar_plot(df, 'CCNL_CODE')#, label_range=30)
b3 = bar_plot(df, 'LEGAL_CATEGORY', rotation=25)#, label_range=30)
b4 = bar_plot(df, 'CCNL_DESCRIPTION', rotation=25, height=600, width=1200)

(b1 + b2 + b3 + b4).cols(1)

:Layout
   .Bars.I   :Bars   [CCNL_LEVEL]   (COUNT)
   .Bars.II  :Bars   [CCNL_CODE]   (COUNT)
   .Bars.III :Bars   [LEGAL_CATEGORY]   (COUNT)
   .Bars.IV  :Bars   [CCNL_DESCRIPTION]   (COUNT)

In [ ]:
b1 = bar_plot(df, 'JOB_FIRST', rotation=25, height=600, width=1200)#, label_range=30)
b2 = bar_plot(df, 'JOB_LAST', rotation=25, height=600, width=1200)

(b1 + b2).cols(1)

:Layout
   .Bars.I  :Bars   [JOB_FIRST]   (COUNT)
   .Bars.II :Bars   [JOB_LAST]   (COUNT)

In [ ]:
b1 = bar_plot(df, 'JOB_SITE', rotation=25, height=800, width=2400)#, label_range=30)
b1

:Bars   [JOB_SITE]   (COUNT)

In [ ]:
h1 = hist_plot(df, "PARTTIME_PERC", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [PARTTIME_PERC]   (Frequency)

In [ ]:
import scipy
print(scipy.__version__)

1.11.4


In [ ]:
h1 = hist_plot(df, "DAYS_ONBENCH_LAST", bins=100, xrange=[], yrange=[], height=400, width=800)
h2 = hist_plot(df, "DAYS_ONBENCH_TOT", bins=100, xrange=[], yrange=[], height=400, width=800)
h3 = hist_plot(df, "DAYS_ONBENCH_PERC", bins=100, xrange=[], yrange=[], height=400, width=800)

(h1 + h2 + h3).cols(1)

:Layout
   .Histogram.I   :Histogram   [DAYS_ONBENCH_LAST]   (Frequency)
   .Histogram.II  :Histogram   [DAYS_ONBENCH_TOT]   (Frequency)
   .Histogram.III :Histogram   [DAYS_ONBENCH_PERC]   (Frequency)

# VACANCY #

In [ ]:
df_vacancy = session.sql(f"""select *
                     from it_discovery.ita_exp_onbench.MPG_IT_ONBENCH_TEST_JOBORDER_CLEANED
                 """).to_pandas()

In [ ]:
keywords = [

    #non utilizzare
    #"curriculum", "cv", "vitae", 

    #generali
    "ricerc", "cerc", "esperienza", "lavor", "occupazion", "posizion", "ruol", "aziend",
    "experience", "job", "position"

    # ruoli
    "addett", "operai", "impiegat", "responsabil", "cassier", "magazzinier",
    "analist", "sviluppator", "developer", "autist", "segretar", "tecnic", "consulent",
    "worker", "manager", "coordinator", "assistant", "engineer", "specialist", "warehouse", "driver", "operator", "supervisor", "administrator", "consultant"

    # titoli di studio
    "diplom", "laure", "master", "istitut", "universit",
    "degree", "master", "university", "school", "college",

    # competenze
    "excel", "office", "ingles", "informatic", "patent",
    "english", "computer", "license"

    # pattern temporali
    #"2019", "2020", "2021", "2022", "2023", "2024", "dal", "al",

    # aziende / enti
    "aziend", "ente", "enti", "srl", "spa", "cooperativ", "comun", "region"
    "regione", "municipality", "company", "organization", "corporation"
]

df_vacancy["IS_VALID_VACANCY"] = df_vacancy["DESCRIPTION"].str.lower().apply(
    lambda x: any(k in x for k in keywords) and len(x) > 50 if isinstance(x, str) else False
)

is_valid_vacancy = np.sum(df_vacancy["IS_VALID_VACANCY"])/df_vacancy.shape[0]

print(f"""Percentage of invalid vacancies (custom keywords matching): {1-is_valid_vacancy:.2%}""")




Percentage of invalid vacancies (custom keywords matching): 38.58%


In [ ]:
general_info(df_vacancy)

Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33548 entries, 0 to 33547
Data columns (total 23 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   JOBORDER_ID_BH           33548 non-null  int32         
 1   SALARY                   33548 non-null  float64       
 2   DATA_INIZIO_VALIDITA     22929 non-null  datetime64[ns]
 3   DATE_LAST_ACTIVE         33548 non-null  datetime64[ns]
 4   JOBORDER_ID_PB           15710 non-null  object        
 5   JOB_TITLE                33548 non-null  object        
 6   CITY                     29321 non-null  object        
 7   REGION                   28136 non-null  object        
 8   SKILLS                   727 non-null    object        
 9   PARTTIME_PERCENT         2246 non-null   object        
 10  TITOLI_RICHIESTI         302 non-null    object        
 11  RICHIESTE_AGGIUNTIVE     465 non-null    object        
 12  REQUISITI                

# COMMENTS on missing/unique values #

32024 vacancies

Comments on candidate missing data (Bullhorn):
-   jobtitle 0%
-   salary 0% 
-   citta 13%
-   regione 16%
-   skills 98%
-   parttime_perc 94%
-   testo_pubblicatione 30%
-   requisiti 47%
-   titoli_richiesti 99%
-   descrizione_uso_interno 30%
-   richieste_aggiuntive 99%
-   description (0.03% missing. this means that skills, testo_pubblicazione, requisiti, titoli_richiesti, descrizione_us_interno are simultaneously missing)

Comments on candidate missing data (PowerBase):
-   job_site 92%
-   job_position 92%
-   parttime_perc 97% missing
-   CCNL level 92% missing
-   CCNL code 92%
-   CCNL description 92%
-   legal_category 92%


Some comments on unique values:
- 12 unique values for titoli_richiesti
- 400 unique regions
- 162 unique CCNL codes
- 196 unique CCNL descriptions
- 103 unique CCNL levels
- 6 unique legal categories

In [ ]:
columns_to_check = ["TITOLI_RICHIESTI", "TESTO_PUBBLICAZIONE", "REQUISITI", "RICHIESTE_AGGIUNTIVE", "DESCRIZIONE_USO_INTERNO"]

unique_limit = 100
# Print unique values for each column
for col in columns_to_check:
    unique_values = df_vacancy[col].sort_values().unique()
    all_values = len(df_vacancy[col])
    nonmissing_values = len(df_vacancy[col].dropna())
    n_unique = len(unique_values)
    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")
    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")

    print(unique_values[:min(unique_limit, n_unique)])
    print("-" * 40 + "\n")



13 unique values in TITOLI_RICHIESTI over 302 total elements:

['diploma' 'diploma,laurea breve' 'diploma,laurea breve,laurea'
 'diploma,non specificato' 'laurea' 'laurea breve' 'laurea breve,laurea'
 'laurea breve,laurea,diploma' 'laurea breve,laurea,master'
 'laurea,laurea breve,master' 'non specificato' 'non specificato,diploma'
 None]
----------------------------------------

21197 unique values in TESTO_PUBBLICAZIONE over 23785 total elements:
(truncated to first 100)
['\n\n\n\n\n\n\n\nexperis è la società it & technology del gruppo manpower.\nle persone e le loro competenze sono il nostro successo e insieme ci poniamo come partner tecnologici dei nostri clienti attraverso servizi e soluzioni.\nvogliamo essere artefici, proprio insieme a te, di un ambiente di lavoro innovativo, stimolante e flessibile, attraverso i nostri valori: people, knowledge e innovation.\nexperis ricerca un\xa0obsolescence engineer\xa0per\xa0potenziare la divisione\xa0aerospazio&difesa\xa0sulla sede di:\xa0

In [ ]:
columns_to_check = ["CCNL_LEVEL", "CCNL_CODE", "LEGAL_CATEGORY", "CCNL_DESCRIPTION"]

unique_limit = 110
# Print unique values for each column
for col in columns_to_check:
    unique_values = df_vacancy[col].sort_values().unique()
    all_values = len(df_vacancy[col])
    nonmissing_values = len(df_vacancy[col].dropna())
    n_unique = len(unique_values)
    print(f"{n_unique} unique values in {col} over {nonmissing_values} total elements:")
    print("" if n_unique < unique_limit else f"(truncated to first {unique_limit})")

    print(unique_values[:min(unique_limit, n_unique)])
    print("-" * 40 + "\n")



108 unique values in CCNL_LEVEL over 2765 total elements:

['1' '1 super q' '130' '140' '2' '3' '3 amministr.' '3 ap liv. 1'
 '3 super' '3/1' '3/2' '3/3' '3a' '4' '4 junior' '4 senior' '4 super'
 '4 super (ex 5)' '4/1' '4/3' '5' '5 (ex 6)' '5/0' '5/1' '5a' '6'
 '6 junior' '6 senior' '6 super' '6 super (ex 7)' '6i' '7' '8' 'a' 'a1'
 'a2' 'a3' 'ae1' 'ae2' 'ap prima' 'ap seconda' 'ap terza' 'area 2'
 'area 3' 'area 3 liv. 1' 'area assistenti' 'area esecutiva'
 'area istruttori' 'area operatori' 'area supporto' 'as1' 'b' 'b classe 1'
 'b super' 'b/2' 'b/3' 'b1' 'b1 super' 'b2' 'b2.' 'b3' 'b3.' 'b4'
 'bs super' 'c' 'c super' 'c/1' 'c/2' 'c/3' 'c0' 'c1' 'c2' 'c2 (ex d1)'
 'c3' 'd' 'd/1' 'd/2' 'd1' 'd1 (ex 6)' 'd1 (ex d2)' 'd1s' 'd2' 'd2 (ex e)'
 'd3' 'e' 'e1' 'e2' 'e3' 'e4' 'eq e funzionari' 'ex 1-2 area' 'f'
 'f 1 posiz.' 'f1' 'f2' 'g' 'g1' 'h' 'i' 'junior' 'op base' 'op centrato'
 'op consolidato' 'op non prof.' 'operat esperti' 'prof.s. e funz.'
 'quadro' None]
---------------------------

In [ ]:
b1 = bar_plot(df_vacancy, 'CITY', rotation=25)#, label_range=30)
b2 = bar_plot(df_vacancy, 'REGION', rotation=25)#, label_range=30)

(b1 + b2).cols(1)

:Layout
   .Bars.I  :Bars   [CITY]   (COUNT)
   .Bars.II :Bars   [REGION]   (COUNT)

In [ ]:
b1 = bar_plot(df_vacancy, 'JOB_SITE', rotation=25)#, label_range=30)
display(b1)


:Bars   [JOB_SITE]   (COUNT)

In [ ]:
h1 = hist_plot(df_vacancy, "SALARY", bins=20, xrange=[], yrange=[], height=400, width=800)

h1

:Histogram   [SALARY]   (Frequency)